In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
# Dummy Values (IQ and Hours Studied before exam for X_train and GPA out of 7 for Y_train, and both are 2D)
X_train=np.array([[115,9],
                  [100,9],
                  [85,7],
                  [92,10],
                  [95,15],
                  [120,25],
                  [66,15],
                  [79,18],
                  [71,18],
                  [93,12]],dtype=np.float32)

Y_train=np.array([[6.5],
                  [6.3],
                  [5.5],
                  [5.75],
                  [6.6],
                  [6.95],
                  [5.85],
                  [5.35],
                  [5.25],
                  [6.0]],dtype=np.float32)

In [3]:
X_train[0,1]

np.float32(9.0)

In [4]:
Y_mean=Y_train.mean()
Y_std=Y_train.std()
X_mean=X_train.mean()
X_std=X_train.std()

In [5]:
Y_train

array([[6.5 ],
       [6.3 ],
       [5.5 ],
       [5.75],
       [6.6 ],
       [6.95],
       [5.85],
       [5.35],
       [5.25],
       [6.  ]], dtype=float32)

In [6]:
# Normalize the values:
X_train=(X_train-X_train.mean(axis=0))/X_train.std(axis=0)
Y_train=(Y_train-Y_train.mean())/Y_train.std()

In [7]:
# Convert it to tensor:
X_train_tensor=torch.from_numpy(X_train)
Y_train_tensor=torch.from_numpy(Y_train)

In [8]:
Y_train_tensor

tensor([[ 0.9157],
        [ 0.5457],
        [-0.9342],
        [-0.4717],
        [ 1.1007],
        [ 1.7481],
        [-0.2867],
        [-1.2117],
        [-1.3967],
        [-0.0092]])

In [9]:
# Define the Linear Regression class:
class LinearRegression(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear=nn.Linear(2,1)

  def forward(self,x):
    out=self.linear(x)
    return out

In [10]:
# Define the loss function and optimizer:
model=LinearRegression()
loss=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-2)
epochs=100

In [11]:
model.state_dict()

OrderedDict([('linear.weight', tensor([[-0.1029, -0.1408]])),
             ('linear.bias', tensor([0.3928]))])

In [12]:
# Train the model:
for epoch in range(epochs):
  # a) Switch the model to training mode:
  model.train()

  # b) Send X_train values one by one:
  out=model(X_train_tensor)

  # c) Find the loss value for each point (True-Predicted)
  model_loss=loss(out,Y_train_tensor)
  optimizer.zero_grad()

  # d) Backward Pass
  model_loss.backward()
  optimizer.step()

  if epoch%5==0:
    print(f"Epochs {epoch+1} / {epochs}: {model_loss.item()}")


Epochs 1 / 100: 1.4144824743270874
Epochs 6 / 100: 1.2542508840560913
Epochs 11 / 100: 1.1107972860336304
Epochs 16 / 100: 0.9847227334976196
Epochs 21 / 100: 0.8759031295776367
Epochs 26 / 100: 0.7834534645080566
Epochs 31 / 100: 0.7058141231536865
Epochs 36 / 100: 0.6409577131271362
Epochs 41 / 100: 0.5866756439208984
Epochs 46 / 100: 0.5408734083175659
Epochs 51 / 100: 0.5017914772033691
Epochs 56 / 100: 0.4681033492088318
Epochs 61 / 100: 0.4388910233974457
Epochs 66 / 100: 0.4135444164276123
Epochs 71 / 100: 0.39164072275161743
Epochs 76 / 100: 0.37284544110298157
Epochs 81 / 100: 0.35685259103775024
Epochs 86 / 100: 0.34336012601852417
Epochs 91 / 100: 0.33206889033317566
Epochs 96 / 100: 0.3226909935474396


In [13]:
model.state_dict()

OrderedDict([('linear.weight', tensor([[0.6473, 0.1762]])),
             ('linear.bias', tensor([-0.0017]))])

In [14]:
# Let's predict out X_train:
model.eval()
with torch.no_grad():
  predictions=model(X_train_tensor)

print(predictions)


tensor([[ 0.7550],
        [ 0.1663],
        [-0.4898],
        [-0.1140],
        [ 0.1722],
        [ 1.4903],
        [-0.9659],
        [-0.3547],
        [-0.6686],
        [-0.0074]])


In [15]:
# Recover original values from normalized values:
predictions_original=predictions*Y_std+Y_mean
print(predictions_original)

tensor([[6.4131],
        [6.0949],
        [5.7402],
        [5.9434],
        [6.0981],
        [6.8106],
        [5.4828],
        [5.8133],
        [5.6436],
        [6.0010]])


In [16]:
# Test on a new student:
new_student=np.array([[120,21],
                        [65,8],
                        [112,9],
                        [65,22]],dtype=np.float32)
new_student_normalized=(new_student-X_mean)/X_std
new_student_tensor=torch.from_numpy(new_student_normalized)

In [17]:
model.eval()
with torch.no_grad():
  pred=model(new_student_tensor)

In [18]:
pred=pred*Y_std+Y_mean
print(pred)

tensor([[6.5076],
        [6.0052],
        [6.4109],
        [6.0379]])
